# LISP benchmark: R vs Python (runtime, memory, scalability)

Reproduces the benchmark in the report. Run the cells top to bottom on a Google Colab **CPU runtime** (Runtime > Change runtime type > CPU) and run nothing else while the benchmark cells are running.

New results are written to `/content/bench_new`, so the published results in `benchmark/results/` are not changed. Absolute times depend on the machine Colab assigns.

## 1. Get the code and data

In [ ]:
!git clone https://github.com/gurpreetk0ur/lisp-python.git
%cd /content/lisp-python

## 2. Record the machine

In [ ]:
!lscpu | grep -E 'Model name|^CPU\(s\)'
!free -h | head -2
!python --version
!R --version | head -1
!lsb_release -ds

## 3. Install the R packages (roughly 5 to 15 minutes)

Uses r2u, which installs CRAN packages as pre-built binaries, then installs localsp, gdverse and sdsfun from GitHub. The published results used localsp 0.2.0, gdverse 1.7 and sdsfun 0.9.0. Later versions may differ, which step 4 checks.

In [ ]:
%%bash
set -e
CODENAME=$(lsb_release -cs)
wget -q https://raw.githubusercontent.com/eddelbuettel/r2u/master/inst/scripts/add_cranapt_${CODENAME}.sh
bash add_cranapt_${CODENAME}.sh > /tmp/r2u_log.txt 2>&1
Rscript -e 'install.packages(c("remotes", "localsp"))'
Rscript -e 'remotes::install_github(c("stscl/sdsfun", "stscl/gdverse", "ausgis/localsp"), upgrade = "never")'
Rscript -e 'for (p in c("localsp","gdverse","sdsfun")) cat(p, as.character(packageVersion(p)), "\n")'

## 4. Check that the installed R code reproduces the reference output

Runs R's `lisp()` on `data/gtc_subset_100.csv` and compares it with `data/r_ground_truth_100.csv`. Continue only if the difference is tiny and there are no label mismatches.

In [ ]:
!Rscript benchmark/bench_r.R data/gtc_subset_100.csv /content/r_check_100.csv 8

import pandas as pd
gt  = pd.read_csv("data/r_ground_truth_100.csv").sort_values("rid").reset_index(drop=True)
new = pd.read_csv("/content/r_check_100.csv").sort_values("rid").reset_index(drop=True)
num = [c for c in gt.columns if c != "rid" and pd.api.types.is_numeric_dtype(gt[c])]
lab = [c for c in gt.columns if c.endswith("_Interaction")]
print("same columns:", set(gt.columns) == set(new.columns))
print("max numeric diff vs reference:", (gt[num] - new[num]).abs().max().max())
print("label mismatches:", sum((gt[c].astype(str) != new[c].astype(str)).sum() for c in lab))

## 5. Validate the Python implementation against the reference output

In [ ]:
!python validation/validate.py

## 6. Run the benchmark

5 sizes (100, 200, 400, 600 and 908 rows; nested random subsets, seed 42). Settings match the validation: threshold 8, 3 to 5 classes, quantile discretisation, R on 1 core. Python takes a few minutes; R took about an hour for one pass in the report. Each finished run is saved straight away.

In [ ]:
!python benchmark/run_benchmarks.py --langs py --repeats 3 --out /content/bench_new

In [ ]:
!python benchmark/run_benchmarks.py --langs r --repeats 1 --out /content/bench_new

## 7. Results: table, correctness at every size, plots

In [ ]:
!python benchmark/analyse_benchmarks.py /content/bench_new

from IPython.display import Image, display
display(Image("/content/bench_new/runtime_vs_size.png"))
display(Image("/content/bench_new/memory_vs_size.png"))